# Think-Vetor: Raciocínio Recorrente Contínuo no Espaço de Embeddings

Bem-vindo ao **Playground Interativo e Demonstrativo do Think-Vetor**!

Este notebook foi desenvolvido para apresentar os conceitos científicos fundamentais, a matemática estrutural e as capacidades de raciocínio da arquitetura **Think-Vetor** (PonderNet + Langevin-Hopfield EBM).

### O que é o Think-Vetor?

Ao contrário dos modelos autoregressivos tradicionais (que geram cadeias de pensamento discretas e verbosas baseadas em tokens), o **Think-Vetor** realiza o processamento lógico cognitivo recursivo inteiramente em seu **espaço latente de embeddings**:
1. **Encoder**: Processa o prompt contextual de entrada e o projeta em estados ocultos iniciais $z_0$.
2. **Ponderação Recorrente (PonderNet)**: Executa loops internos dinâmicos sobre $z_k$, avaliando a probabilidade de parada (*halting*) a cada passo.
3. **Langevin-Hopfield EBM (Memórias Baseadas em Energia)**: Refina os estados latentes a cada passo em direção a atratores estáveis que minimizam a energia livre das memórias cognitivas:
   $$E(z) = -\frac{1}{\beta} \log \sum_{i=1}^{M} \exp(\beta M_i^T z) + \frac{1}{2} \|z\|^2$$
4. **Decoder**: Recebe o estado latente condensado e livre de ruído ($z_K$) e gera autoregressivamente **apenas a resposta terminal** (ex: `"Alice"`), economizando até 2.67x em tokens de contexto físico de rede e de memória.

## 1. Configuração do Ambiente e Clonagem do Repositório

Se você estiver executando este notebook no **Google Colab**, a célula abaixo clonará automaticamente o repositório oficial do projeto, adicionará as dependências e ajustará os caminhos de importação do Python.

In [ ]:
import os
import sys

# Detectar se está rodando no Google Colab
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    print("[INFO] Executando no Google Colab. Configurando repositório...")
    if not os.path.exists("crom-microllm-think-vetor"):
        !git clone https://github.com/MrJc01/crom-microllm-think-vetor.git
        %cd crom-microllm-think-vetor
    sys.path.append(os.getcwd())
else:
    print("[INFO] Executando em ambiente local. Verificando diretório...")
    # Garantir que estamos no diretório correto local
    if os.path.exists("src"):
        sys.path.append(os.getcwd())
        print(f"  Ambiente local configurado em: {os.getcwd()}")
    else:
        print("[AVISO] Pasta 'src' não encontrada. Certifique-se de executar o Jupyter a partir da raiz do projeto.")

## 2. Carregamento Automático do Checkpoint e Tokenizer

Carregamos os pesos do modelo treinado. Nossa rotina inteligente analisa a estrutura do dicionário de pesos (`state_dict`) para reconstruir dinamicamente a topologia correta do modelo (camadas, RoPE, memórias Hopfield) e selecionar o tokenizer correto.

In [ ]:
import torch
from interactive_playground import detect_architecture_and_tokenizer

# Definir o dispositivo padrão
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Usando dispositivo: {device.type.upper()}")

# Caminhos possíveis para encontrar o checkpoint do modelo lógico
checkpoint_options = [
    "checkpoints/think_vetor_logic_exported/weights.pt",
    "checkpoints/think_vetor_hybrid_best.pt",
    "think_vetor_hybrid_best.pt"
]

checkpoint_path = None
for path in checkpoint_options:
    if os.path.exists(path):
        checkpoint_path = path
        break

if checkpoint_path is None:
    raise FileNotFoundError(
        "[ERRO] Nenhum checkpoint Think-Vetor encontrado. Certifique-se de executar export_model.py ou ter treinado um modelo primeiro."
    )

# Carregamento agnóstico do modelo e tokenizer correspondente
model, tokenizer, is_causal_coconut = detect_architecture_and_tokenizer(checkpoint_path)
model = model.to(device)

## 3. Playground Interativo

Execute a célula abaixo para abrir o console interativo de inferência. 

### Exemplos de prompts para digitar:
1. **Transitividade Relacional Clássica**:
   * *"Alice is older than Bob. Bob is older than Charlie. Who is older, Alice or Charlie?"*
2. **Transitividade com Erratas e Correção Retrospectiva**:
   * *"Alice is taller than Bob. Bob is taller than Charlie. Wait, Alice is shorter than Bob instead. Who is taller, Bob or Charlie?"*
3. **Word Problem Aritmético Aprimorado**:
   * *"Alice has 32 cards. Bob has 18. Who has more?"*

In [ ]:
from interactive_playground import run_inference

print("=== PLAYGROUND INTERATIVO THINK-VETOR ===")
print("Digite o seu prompt (ou digite 'sair' ou 'exit' para encerrar):")

while True:
    try:
        user_prompt = input("\nPrompt > ").strip()
        if user_prompt.lower() in ['sair', 'exit']:
            print("Playground encerrado. Obrigado!")
            break
        if not user_prompt:
            continue
            
        # Executa inferência capturando halting e passos lógicos
        run_inference(model, tokenizer, user_prompt, is_causal_coconut, device)
    except KeyboardInterrupt:
        print("\nPlayground interrompido.")
        break
    except Exception as e:
        print(f"[ERRO] Ocorreu uma falha no processamento: {e}")